In [ ]:
# conda create --prefix ./envs -c conda-forge dask-core=2023.4.0 datashader=0.14.4 flask=2.2.3 geopandas=0.12.2 ipykernel=6.19.2 matplotlib=3.7.1 numpy=1.23.5 pandas=2.0.0 python=3.9.16 pygeos=0.14 flask-cors=6.0.1 geopy=2.4.1 osmnx=1.9.4 pyrosm=0.6.2 osmium-tool=1.16.0 
# for windows extra: cudatoolkit=11.2.2 cudnn=8.1.0.77

# pip install spatialpandas==0.4.7 opencv-python==4.7.0.72 pyarrow==11.0.0 osmium==3.6.0
# for mac: pip install tensorflow-macos
# for windows: pip install tensorflow-gpu==2.8.0

In [12]:
# change current path to backend. cur dir + backend

In [1]:
import pandas as pd
import geopandas as gpd
import spatialpandas as sp
import datashader as ds
import numpy as np
import pyarrow as pa
import matplotlib.pyplot as plt
import pygeos
import math
import os
os.environ['USE_PYGEOS'] = '0'
import time
import warnings
import cv2
import glob
import dask

from shapely.geometry import box
from pyproj import Transformer
from spatialpandas import GeoDataFrame

import datashader as ds
from datashader.core import bypixel

C:\Users\qshah\AppData\Local\Temp\ipykernel_14000\3267991667.py:2: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


In [2]:
# os.chdir("./backend")

In [3]:
transformer = Transformer.from_crs(3395, 4326)
invtransformer = Transformer.from_crs(4326,3395)

In [4]:
def get_flat_coords_offset_arrays(arr):
    """
    Version for MultiPolygon data
    """
    # explode/flatten the MultiPolygons
    arr_flat, part_indices = pygeos.get_parts(arr, return_index=True)
    # the offsets into the multipolygon parts
    offsets1 = np.insert(np.bincount(part_indices).cumsum(), 0, 0)

    # explode/flatten the Polygons into Rings
    arr_flat2, ring_indices = pygeos.geometry.get_rings(arr_flat, return_index=True)
    # the offsets into the exterior/interior rings of the multipolygon parts 
    offsets2 = np.insert(np.bincount(ring_indices).cumsum(), 0, 0)

    # the coords and offsets into the coordinates of the rings
    coords, indices = pygeos.get_coordinates(arr_flat2, return_index=True)
    offsets3 = np.insert(np.bincount(indices).cumsum(), 0, 0)
    
    return coords, offsets1, offsets2, offsets3

def spatialpandas_from_pygeos(arr):
    coords, offsets1, offsets2, offsets3 = get_flat_coords_offset_arrays(arr)
    coords_flat = coords.ravel()
    offsets3 *= 2
    
    # create a pyarrow array from this
    _parr3 = pa.ListArray.from_arrays(pa.array(offsets3), pa.array(coords_flat))
    _parr2 = pa.ListArray.from_arrays(pa.array(offsets2), _parr3)
    parr = pa.ListArray.from_arrays(pa.array(offsets1), _parr2)
    
    return sp.geometry.MultiPolygonArray(parr)

def polygons(self, source, geometry, agg=None):
    from datashader.glyphs import PolygonGeom
    from datashader.reductions import any as any_rdn
    from spatialpandas import GeoDataFrame
    from spatialpandas.dask import DaskGeoDataFrame
    if isinstance(source, DaskGeoDataFrame):
        # Downselect partitions to those that may contain polygons in viewport
        x_range = self.x_range if self.x_range is not None else (None, None)
        y_range = self.y_range if self.y_range is not None else (None, None)
        source = source.cx_partitions[slice(*x_range), slice(*y_range)]
    elif isinstance(source, gpd.GeoDataFrame):
        # Downselect actual rows to those for which the polygon is in viewport
        x_range = self.x_range if self.x_range is not None else (None, None)
        y_range = self.y_range if self.y_range is not None else (None, None)
        source = source.cx[slice(*x_range), slice(*y_range)]
        # Convert the subset to ragged array format of spatialpandas
        geometries = spatialpandas_from_pygeos(source.geometry.array.data)
        source = pd.DataFrame(source)
        source["geometry"] = geometries
    elif not isinstance(source, GeoDataFrame):
        raise ValueError(
            "source must be an instance of spatialpandas.GeoDataFrame or \n"
            "spatialpandas.dask.DaskGeoDataFrame.\n"
            "  Received value of type {typ}".format(typ=type(source)))

    if agg is None:
        agg = any_rdn()
    glyph = PolygonGeom(geometry)
    return bypixel(source, self, glyph, agg)

ds.Canvas.polygons = polygons
cvs = ds.Canvas(plot_width=256, plot_height=256)

In [5]:
def deg2num(lat_deg, lon_deg, zoom):
    lat_rad = math.radians(lat_deg)
    n = 2.0 ** zoom
    xtile = ((lon_deg + 180.0) / 360.0 * n)
    ytile = ((1.0 - math.asinh(math.tan(lat_rad)) / math.pi) / 2.0 * n)
    return (xtile, ytile)

def num2deg(xtile, ytile, zoom):
    n = 2.0 ** zoom
    lon_deg = xtile / n * 360.0 - 180.0
    lat_rad = math.atan(math.sinh(math.pi * (1 - 2 * ytile / n)))
    lat_deg = math.degrees(lat_rad)
    return (lat_deg, lon_deg)

In [6]:
def elevation(filtered, bbox):
    proxy = pd.DataFrame({'height': 0, 'geometry': bbox}, index=[len(filtered)])
    proxy = gpd.GeoDataFrame(proxy)
    proxy.crs = '3395'

    clipped = gpd.clip(filtered, proxy)
    intersection = pd.concat([proxy, clipped], ignore_index=True)
    intersection = intersection[intersection.geom_type.isin(['Polygon', 'MultiPolygon'])]
    if len(intersection) > 0:
        intersection = sp.GeoDataFrame(intersection)
        values = cvs.polygons(intersection, geometry='geometry', agg=ds.max("height"))
    else:
        values = np.zeros((256,256))
    values = np.flipud(values)
    return values

def create_image(values, i, j, zoom, max_height, outputfolder):
    filename = '%s/%d/%d/%d.png'%(outputfolder,zoom,i,j)
    success = cv2.imwrite(filename, 255.0 * (values / max_height))
    if not success:
        raise Exception("Could not write image")
        
@dask.delayed
def compute_tile(gdf, i, j, zoom, max_height, outputfolder):
    bb0 = num2deg(i,j,zoom)
    bb1 = num2deg(i+1,j+1,zoom)
    bb0 = invtransformer.transform(bb0[0],bb0[1])
    bb1 = invtransformer.transform(bb1[0],bb1[1])
    bbox = box(bb0[0],bb0[1],bb1[0],bb1[1])
#     filtered = gdf.cx[bb0[0]:bb1[0],bb0[1]:bb1[1]]
    filtered = gdf.loc[gdf.sindex.intersection(bbox.bounds)]
    
    if len(filtered) > 0:
        values = elevation(filtered, bbox)
        create_image(values, i, j, zoom, max_height, outputfolder)
        
def compute_all(gdf, zoom, max_height, outputfolder):
    bounds = gdf.total_bounds
    lat0,lng0 = transformer.transform(bounds[0],bounds[1])
    lat1,lng1 = transformer.transform(bounds[2],bounds[3])
    coord0 = deg2num(lat0,lng0,zoom)
    coord1 = deg2num(lat1,lng1,zoom)
    bottomleft = [min(coord0[0],coord1[0]),min(coord0[1],coord1[1])]
    topright = [max(coord0[0],coord1[0]),max(coord0[1],coord1[1])]
    
    # Create folders (serial)
    for i in range(math.floor(bottomleft[0]),math.ceil(topright[0])):
        folder = '%s/%d/%d/'%(outputfolder,zoom,i)
        if not os.path.exists(folder):
            os.makedirs(folder)
    
    delayed = []
    for i in range(math.floor(bottomleft[0]),math.ceil(topright[0])):
        for j in range(math.floor(bottomleft[1]),math.ceil(topright[1])):
            ddelayed = compute_tile(gdf, i, j, zoom, max_height, outputfolder)
            delayed.append(ddelayed)
    dask.compute(*delayed)

In [7]:
def elevation(filtered, bbox):
    proxy = pd.DataFrame({'height': 0, 'geometry': bbox}, index=[len(filtered)])
    proxy = gpd.GeoDataFrame(proxy)
    proxy.crs = '3395'

    clipped = gpd.clip(filtered, proxy)
    intersection = pd.concat([proxy, clipped], ignore_index=True)
    intersection = intersection[intersection.geom_type.isin(['Polygon', 'MultiPolygon'])]
    if len(intersection) > 0:
        intersection = sp.GeoDataFrame(intersection)
        values = cvs.polygons(intersection, geometry='geometry', agg=ds.max("height"))
    else:
        values = np.zeros((256,256))
    values = np.flipud(values)
    return values

def create_image(values, i, j, zoom, max_height, outputfolder):
    filename = '%s/%d_%d_%d.png'%(outputfolder,zoom,i,j)
    success = cv2.imwrite(filename, 255.0 * (values / max_height))
    if not success:
        raise Exception("Could not write image")
        
# @dask.delayed
def compute_tile(gdf, i, j, zoom, max_height, outputfolder):
    bb0 = num2deg(i,j,zoom)
    bb1 = num2deg(i+1,j+1,zoom)
    bb0 = invtransformer.transform(bb0[0],bb0[1])
    bb1 = invtransformer.transform(bb1[0],bb1[1])
    bbox = box(bb0[0],bb0[1],bb1[0],bb1[1])
#     filtered = gdf.cx[bb0[0]:bb1[0],bb0[1]:bb1[1]]
    filtered = gdf.loc[gdf.sindex.intersection(bbox.bounds)]
    
    if len(filtered) > 0:
        values = elevation(filtered, bbox)
        create_image(values, i, j, zoom, max_height, outputfolder)

    else:
        print(f"No data for tile {zoom}/{i}/{j}")

In [8]:
# Read a GeoJSON file
gdf = gpd.read_file("./data/served/baselayer-0_buildings.geojson")
min_lon, min_lat, max_lon, max_lat = gdf.total_bounds
outputfolder = "./data/served/baselayer-0_buildings_raster/"

os.makedirs(outputfolder, exist_ok=True)

z = 16

# (top-left)
x0, y0 = deg2num(max_lat, min_lon, z)
# (bottom-right)
x1, y1 = deg2num(min_lat, max_lon, z)

# Tile index ranges (inclusive)
min_x = math.floor(x0)   # left
max_x = math.ceil(x1)    # right
min_y = math.floor(y0)   # top
max_y = math.ceil(y1)    # bottom

# change gdf crs to 3395
gdf = gdf.to_crs(epsg=3395)

delayed = []
for i in range(min_x, max_x + 1):
    for j in range(min_y, max_y + 1):
        ddelayed = compute_tile(gdf, i, j, z, 550, outputfolder)
        delayed.append(ddelayed)
    
dask.compute(*delayed);

No data for tile 16/16812/24359
No data for tile 16/16812/24360
No data for tile 16/16813/24359
No data for tile 16/16813/24360
No data for tile 16/16814/24360
No data for tile 16/16815/24360
No data for tile 16/16816/24359
No data for tile 16/16816/24360
No data for tile 16/16817/24358
No data for tile 16/16817/24359
No data for tile 16/16817/24360
No data for tile 16/16818/24353
No data for tile 16/16818/24354
No data for tile 16/16818/24355
No data for tile 16/16818/24356
No data for tile 16/16818/24357
No data for tile 16/16818/24358
No data for tile 16/16818/24359
No data for tile 16/16818/24360
